# 02 · Modeling and evaluation

CRISP-DM phases 4 and 5.

The full pipeline lives in `nyctaxi.train` and is what actually produces the
shipped artifact — this notebook inspects the *result* of a training run rather
than reimplementing it, so there is one source of truth for how the model was
built.

```bash
python -m nyctaxi.train --sample-frac 0.06
python -m nyctaxi.evaluate
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nyctaxi.clean import clean, haversine_km
from nyctaxi.config import get_config
from nyctaxi.data.loader import load

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18, "font.size": 9,
})
ACCENT, GOOD = "#e5a000", "#128a5b"
cfg = get_config()
print("config loaded:", cfg.project_name)

In [ ]:
from nyctaxi import registry
from nyctaxi.metrics import interval_coverage, score_all

bundle = registry.load()
meta, metrics = bundle["metadata"], bundle["metrics"]

print(f"version      {bundle['version']}")
print(f"trained      {meta['trained_at'][:19]}  (commit {meta['git_sha']})")
print(f"source       {meta['data_source']}  (zone resolution: {meta['zone_resolution']})")
print(f"rows         {meta['rows_clean']:,} clean  "
      f"({meta['rows_train']:,} train / {meta['rows_valid']:,} valid)")
print(f"features     {len(meta['features'])}")
print(f"best round   {meta['best_iteration']:,}")

## The ladder

Each rung exists so the next one has to earn its place. A single gradient
boosting score in isolation would tell us nothing about whether the complexity
was justified.

In [ ]:
board = pd.DataFrame(metrics["leaderboard"]).sort_values("rmsle")
display(board[["model", "rmsle", "mae_s", "rmse_s", "r2", "train_seconds"]]
        .style.format({"rmsle": "{:.4f}", "mae_s": "{:.0f}", "rmse_s": "{:.0f}", "r2": "{:.3f}"})
        .background_gradient(subset=["rmsle"], cmap="RdYlGn_r"))

fig, ax = plt.subplots(figsize=(7, 3))
colors = [ACCENT if m == metrics["production_model"] else "#c9ccd2" for m in board["model"]]
ax.barh(board["model"].str.replace("_", " "), board["rmsle"], color=colors)
ax.invert_yaxis()
ax.set(title="Validation RMSLE (lower is better)", xlabel="RMSLE")
plt.tight_layout()

## Is the confidence band honest?

The single most important check on the quantile models. A nominal-80% interval
that actually covers 50% of trips is worse than showing no interval at all,
because it invites misplaced trust.

In [ ]:
preds = pd.read_parquet(
    cfg.paths.resolve("models") / bundle["version"] / "validation_predictions.parquet"
)

overall = score_all(preds["y_true"].to_numpy(), preds["y_pred"].to_numpy())
cov = interval_coverage(preds["y_true"].to_numpy(),
                        preds["q10"].to_numpy(), preds["q90"].to_numpy())

print("overall:", overall)
print(f"\ncoverage {cov['coverage_pct']:.2f}% vs 80% nominal "
      f"(median band {cov['median_width_s'] / 60:.1f} min)")

gap = abs(cov["coverage_pct"] - 80)
print("verdict:", "within tolerance — readable at face value" if gap <= 5
      else "outside tolerance — the band needs work")

In [ ]:
buckets = pd.cut(preds["haversine_km"], [0, 1, 2, 5, 10, 20, 1000],
                 labels=["<1", "1-2", "2-5", "5-10", "10-20", ">20"])
inside = (preds["y_true"] >= preds["q10"]) & (preds["y_true"] <= preds["q90"])
by_dist = inside.groupby(buckets, observed=True).mean() * 100

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(range(len(by_dist)), by_dist.to_numpy(),
       color=[GOOD if abs(v - 80) <= 5 else "#c2410c" for v in by_dist])
ax.axhline(80, ls="--", color="#14171c", lw=1.2, label="80% nominal")
ax.set_xticks(range(len(by_dist)))
ax.set_xticklabels(by_dist.index.astype(str))
ax.set(title="Interval coverage by trip length", xlabel="distance (km)",
       ylabel="coverage (%)", ylim=(0, 105))
ax.legend(frameon=False)
plt.tight_layout()

## Residuals and error structure

Residuals should centre on zero with no trend against the prediction. Error is
then sliced by hour and distance, because an average hides exactly the cases a
rider cares about — rush hour and airport runs.

In [ ]:
resid = (preds["y_pred"] - preds["y_true"]) / 60
lo, hi = np.percentile(resid, [0.5, 99.5])

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))
axes[0].scatter(preds["y_pred"] / 60, resid, s=3, alpha=0.1, color=ACCENT, linewidths=0)
axes[0].axhline(0, ls="--", color="#14171c", lw=1)
axes[0].set(title="Residuals vs predicted", xlabel="predicted (min)", ylabel="error (min)",
            ylim=(lo, hi))

axes[1].hist(resid[(resid > lo) & (resid < hi)], bins=70, color=ACCENT)
axes[1].axvline(0, ls="--", color="#14171c", lw=1)
axes[1].set(title="Residual distribution", xlabel="error (min)")

err = preds.assign(abs_err=resid.abs()).groupby("hour")["abs_err"].mean()
axes[2].bar(err.index, err.to_numpy(), color=GOOD)
axes[2].set(title="MAE by departure hour", xlabel="hour", ylabel="MAE (min)")
plt.tight_layout()

print(f"median residual {resid.median():+.2f} min  (a large bias here would mean "
      f"systematic over- or under-estimation)")

## What the model leans on

In [ ]:
imp = pd.DataFrame(metrics["feature_importance"]).head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(imp["feature"], imp["gain"], color=ACCENT)
ax.set(title="Feature importance (LightGBM split gain)", xlabel="gain")
plt.tight_layout()

## Time split vs random split

Reported results use a **time-based** split, which mirrors deployment. A random
split shuffles days together, letting the model see traffic from the very days
it is scored on. The gap below is how much optimism that would have bought.

In [ ]:
sc = metrics["split_comparison"]
print(f"time split   {sc['time_split']['rmsle']:.4f}   <- reported")
print(f"random split {sc['random_split']['rmsle']:.4f}")
print(f"optimism     {sc['time_split']['rmsle'] - sc['random_split']['rmsle']:+.4f} RMSLE")
print()
print(sc["note"])

## Live inference

The same artifact the API serves, exercised directly — the check that training
and serving have not drifted apart.

In [ ]:
from datetime import datetime
from nyctaxi.api.predict import PredictionService

svc = PredictionService()
TIMES_SQUARE, JFK = (40.7580, -73.9855), (40.6413, -73.7781)

for label, when in [("weekday 05:00", datetime(2016, 3, 2, 5)),
                    ("weekday 17:30", datetime(2016, 3, 2, 17, 30))]:
    out = svc.predict(TIMES_SQUARE, JFK, departure=when, passengers=2)
    d = out["duration"]
    print(f"{label}:  {d['point_s'] / 60:5.1f} min   "
          f"band {d['p10_s'] / 60:.0f}-{d['p90_s'] / 60:.0f} min   "
          f"fare ${out['fare']['total']:.2f}")

In [ ]:
curve = svc.hourly_curve(TIMES_SQUARE, JFK)
pts = pd.DataFrame(curve["points"])

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.fill_between(pts["hour"], pts["p10_s"] / 60, pts["p90_s"] / 60, color=ACCENT, alpha=0.22,
                label="P10-P90")
ax.plot(pts["hour"], pts["p50_s"] / 60, color=ACCENT, lw=2, label="estimate")
ax.set(title="Times Square to JFK, by departure hour", xlabel="hour", ylabel="minutes")
ax.legend(frameon=False)
plt.tight_layout()

print(f"quickest {curve['best_hour']}:00, slowest {curve['worst_hour']}:00")

## Conclusion

LightGBM roughly halves the RMSLE of the physics baseline and cuts typical
error from over 7 minutes to about 3¼, on an honest time-based split. The
P10–P90 band covers close to its nominal 80%, so the uncertainty shown in the
app means what it says.

The clearest remaining headroom is data, not modelling: live traffic, weather,
and — on the TLC path — true coordinates instead of zone-sampled ones.

See [`06-deployment.md`](../docs/06-deployment.md) for how this is served.